# Ontario FRI <-> Canada NFI Species Composition Converter

This interactive notebook demonstrates how to use the `fri_nfi_tool` API to:
1. Convert individual species composition strings between Ontario FRI and Canada NFI formats.
2. Inspect and query the underlying 107-species reference mapping.
3. Process CSV Value Attribute Tables (VAT) and preview enriched tabular data with Pandas.
4. Save converted tables ready for GIS joins.

## 1. Setup & Imports

Import the core conversion functions. If running from within the `fri_nfi_tool` directory or with the package installed, these functions are directly importable.

In [ ]:
from pathlib import Path
import pandas as pd

# Support running whether the notebook is executed from inside fri_nfi_tool/ or parent directory
try:
    from fri_nfi_tool import (
        fri_to_nfi,
        nfi_to_fri,
        parse_fri_spp_string,
        convert_fri_attribute_table,
        FRI_TO_NFI_CODES,
        NFI_TO_FRI_CODES,
    )
except ImportError:
    from converter import (
        fri_to_nfi,
        nfi_to_fri,
        parse_fri_spp_string,
        convert_fri_attribute_table,
        FRI_TO_NFI_CODES,
        NFI_TO_FRI_CODES,
    )

print("FRI <-> NFI converter imported successfully!")

## 2. Converting Individual Species Strings

In the Ontario Forest Resource Inventory (FRI), species composition is encoded as a compressed string of species codes and percentages (e.g., `Cw  50Sb  40La  10`).

- **`fri_to_nfi(fri_str)`**: Parses the FRI string and converts each species into an NFI standard dictionary containing `genus`, `species`, `variety`, and `percent`.
- **`nfi_to_fri(nfi_entries)`**: Reconstructs the compressed FRI string.

In [ ]:
sample_fri = "Cw  50Sb  40La  10"  # 50% Cedar, 40% Black Spruce, 10% Tamarack

# 1. Convert FRI -> NFI
nfi_result = fri_to_nfi(sample_fri)

# Display as a clean DataFrame
df_nfi = pd.DataFrame(nfi_result)
display(df_nfi)

# 2. Round-trip conversion: NFI -> FRI
reconstructed_fri = nfi_to_fri(nfi_result)
print(f"Original FRI:      '{sample_fri}'")
print(f"Reconstructed FRI: '{reconstructed_fri}'")

### Handling Varied Formats & Edge Cases

The parser handles pure stands, varied whitespace, and empty/non-forest cells gracefully:

In [ ]:
test_cases = [
    "Sb 100",              # Pure Black Spruce
    "Pt  70Bw  20Sb  10",   # Hardwood mixed stand
    "Pw 60Pr 40",           # White pine & red pine
    " ",                    # Non-forest / water / unclassified
]

for s in test_cases:
    entries = fri_to_nfi(s)
    summary = " | ".join(
        [f"{e['genus']} {e['species']} ({int(e['percent'])}%)" for e in entries]
    ) if entries else "[Non-forested / Empty]"
    print(f"{repr(s):<22} -> {summary}")

## 3. Querying Species Reference Codes

You can inspect the 107 species registered in Ontario and their official NFI mappings:

In [ ]:
# Convert reference dictionary to a searchable DataFrame
ref_df = pd.DataFrame([
    {"FRI_Code": k, "NFI_Genus": v[0], "NFI_Species": v[1], "Variety": v[2]}
    for k, v in FRI_TO_NFI_CODES.items()
])

# Search by genus (e.g., PICE = Picea / Spruce, PINU = Pinus / Pine, ACER = Acer / Maple)
ref_df[ref_df["NFI_Genus"].isin(["PICE", "PINU", "ACER"])].head(15)

## 4. Processing CSV Attribute Tables

Here we process an entire raster Value Attribute Table (VAT) CSV.
The function parses the species string for each polygon/stand and augments the table with:
- `nfi_summary`: Human-readable summary (e.g. `THUJ OCC 50% | PICE MAR 40% | LARI LAR 10%`)
- `leading_spp`: Dominant tree species
- `nfi_json`: Serialized JSON array of all species
- `spp1_genus`, `spp1_species`, `spp1_percent`, ...: Flattened columns for direct GIS joins

In [ ]:
# Determine file paths relative to notebook
nb_dir = Path.cwd()
if (nb_dir / "sample_data").is_dir():
    sample_dir = nb_dir / "sample_data"
elif (nb_dir / "fri_nfi_tool" / "sample_data").is_dir():
    sample_dir = nb_dir / "fri_nfi_tool" / "sample_data"
else:
    sample_dir = Path("./sample_data").resolve()

input_csv = sample_dir / "sample_attr_table.csv"
output_csv = sample_dir / "output_nfi_attr.csv"

# View the raw input table
raw_df = pd.read_csv(input_csv)
print(f"Raw input table: {len(raw_df)} rows")
display(raw_df.head())

In [ ]:
# Run the attribute table conversion
converted_rows = convert_fri_attribute_table(
    input_csv_path=input_csv,
    output_csv_path=output_csv,
    spp_column_name="OSPCOMP"  # Auto-detected if omitted
)

# Load and inspect the converted output table
output_df = pd.read_csv(output_csv)
print(f"Converted table saved to: {output_csv}")

# Display key converted columns
preview_cols = [
    "Value", "OSPCOMP", "leading_spp", "nfi_summary",
    "spp1_genus", "spp1_species", "spp1_percent",
    "spp2_genus", "spp2_species", "spp2_percent"
]
display(output_df[preview_cols].head(10))

## 5. In-Memory Pandas DataFrame Conversion

If you already have a Pandas DataFrame in your notebook workflow, you can convert species strings on-the-fly using `.apply()`:

In [ ]:
def summarize_spp(spp_str):
    entries = fri_to_nfi(spp_str, ignore_unknown=True)
    if not entries:
        return ""
    return " | ".join([f"{e['genus']} {e['species']} {int(round(e['percent']))}%" for e in entries])

def get_leading_genus(spp_str):
    entries = fri_to_nfi(spp_str, ignore_unknown=True)
    if not entries:
        return None
    return max(entries, key=lambda x: x["percent"])["genus"]

# Apply directly to DataFrame
df_interactive = raw_df.copy()
df_interactive["nfi_summary"] = df_interactive["OSPCOMP"].apply(summarize_spp)
df_interactive["leading_genus"] = df_interactive["OSPCOMP"].apply(get_leading_genus)

display(df_interactive[["Value", "OSPCOMP", "leading_genus", "nfi_summary"]].head(10))

# Quick distribution of leading tree genera in the dataset
print("\nStand counts by leading genus:")
print(df_interactive["leading_genus"].value_counts(dropna=False))